# **Cross-Validation in Machine Learning**

Prerequisites: Statistics primer (CLT), Bias-Variance Tradeoff.

## **Introduction**

Cross-validation is a statistical method used to evaluate the performance of a machine learning model. It ensures that the model performs well not only on the training data but also on unseen data, thus preventing overfitting and underfitting.

##
---

## **Key Characteristics**

1. **Purpose**: Cross-validation is an **evaluation technique**, not a feature engineering or preprocessing method.
2. **Main Idea**: Split the dataset into training and testing sets multiple times to evaluate model performance more reliably.
3. **Advantage**: Provides a better estimate of a model’s generalization error compared to a single train-test split.

![Cross validation idea.png](../images/cross_validation.png)

##
---

## **Why Use Cross-Validation?**

- **Model Validation**: Provides a reliable estimate of model performance.

- **Reduces Overfitting**: Ensures the model is not overfitted to the training data.

- **Model Selection**: Helps compare the performance of different algorithms.

- **Improved Generalization**: Evaluates how well the model performs on unseen data.

##
---

## **How It Works**

The core idea of cross-validation is to divide the dataset into subsets (folds) and perform multiple training and testing cycles to ensure the model is validated on different portions of the data.


##
---

## **Types of Cross-Validation**

### 1. **K-Fold Cross-Validation**

A single train/test split gives one noisy estimate of generalization
error. k-fold CV partitions data into $k$ folds, trains on $k-1$, tests on
the held-out fold, cycles through all $k$, and averages:

$$\widehat{\text{Err}}_{CV} = \frac1k\sum_{i=1}^k \text{Err}_i$$
By the Central Limit Theorem (Probability primer §6), the variance of this
**average** of $k$ (roughly independent) error estimates is lower than the
variance of any single estimate — $\text{Var}(\widehat{\text{Err}}_{CV})\approx\text{Var}(\text{Err}_i)/k$ 
(approximately, since folds aren't fully independent — they share
overlapping training data). This is the precise mechanical reason k-fold
CV is preferred over one split, not just "it uses more data."

In [1]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

# Load dataset
data = load_iris()
X, y = data.data, data.target

# K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier()

# Evaluate model
scores = cross_val_score(model, X, y, cv=kf)
print("Accuracy for each fold:", scores)
print("Average Accuracy:", scores.mean())

Accuracy for each fold: [1.         0.96666667 0.93333333 0.93333333 0.96666667]
Average Accuracy: 0.9600000000000002


#### Worked Numerical Example - compute k-fold mean and variance by hand

Tiny dataset, 6 points, 3-fold CV (fold size 2). Suppose the per-fold
squared errors, after fitting/testing each fold, are:
Fold 1: $\{1.2, 0.8\}$ → mean $=1.0$
Fold 2: $\{2.1, 1.9\}$ → mean $=2.0$
Fold 3: $\{0.5, 0.7\}$ → mean $=0.6$

$$\widehat{\text{Err}}_{CV} = \frac{1.0+2.0+0.6}{3} = \frac{3.6}{3}=1.2$$
$$\text{Var across folds} = \frac13\big[(1.0-1.2)^2+(2.0-1.2)^2+(0.6-1.2)^2\big]=\frac13[0.04+0.64+0.36]=\frac{1.04}{3}\approx0.347$$
Standard error of the CV estimate $\approx\sqrt{0.347/3}\approx0.34$ — this
standard-error quantity (not just the point estimate 1.2) is what should
be reported, since it tells you whether a difference between two models'
CV scores is likely meaningful or just noise (direct link to
`05_Model_Evaluation/07_validation_strategies.ipynb`'s paired-test content
in `16-Model-Evaluation-Additions.md`).

In [1]:
## verify the hand-worked fold statistics

import numpy as np

fold_errors = [[1.2,0.8],[2.1,1.9],[0.5,0.7]]
fold_means = [np.mean(f) for f in fold_errors]
cv_estimate = np.mean(fold_means)
cv_variance = np.var(fold_means, ddof=0)
print(fold_means, cv_estimate, cv_variance)   # [1.0, 2.0, 0.6]  1.2  0.3467
print("SE:", np.sqrt(cv_variance/3))          # 0.340

[1.0, 2.0, 0.6] 1.2 0.3466666666666667
SE: 0.33993463423951903


###
---

- **2. Choosing $k$ — the bias-variance tradeoff of $k$ itself**

- Small $k$ (e.g. $k=2$): each training fold uses only half the data →
  the per-fold model is trained on less data than the full dataset would
  give → **pessimistically biased** estimate of the error you'd get from
  a model trained on all $n$ points; but folds are quite different from
  each other → lower variance in the final average.
- Large $k$ (e.g. $k=n$, "Leave-One-Out"): each training fold uses almost
  all the data → nearly unbiased estimate of the full-data model's error;
  but the $n$ per-fold models are nearly identical (trained on
  nearly-identical data) and highly correlated → **higher variance** in
  the average (paradoxically) despite lower bias per fold.
- $k=5$ or $k=10$ is the standard practical compromise.

---

### 2. **Stratified K-Fold Cross-Validation**

- Similar to K-Fold but preserves the proportion of classes (target variable) in each fold.

- Useful for imbalanced datasets.

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 1.0
Accuracy: 0.9666666666666667
Accuracy: 0.9333333333333333
Accuracy: 0.9666666666666667
Accuracy: 0.9


###
---

### 3. **Leave-One-Out Cross-Validation (LOOCV)**

- Uses one data point as the validation set and the rest for training.

- Repeated for every data point.

- Computationally expensive for large datasets.

In [4]:
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print("True:", y_test, "Predicted:", predictions)

True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]
True: [0] Predicted: [0]


###
---

### 4. **Hold-Out Validation**

- The dataset is split into separate training and testing sets.

- Simpler but prone to bias if the split isn't representative.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 1.0


###
---

### 5. **Time Series Cross-Validation**

- Used for time-dependent data.

- Splits the data sequentially to respect temporal order.

- Avoids "data leakage" from future observations.

In [6]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)

for train_index, test_index in tscv.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.2972972972972973
Accuracy: 0.6486486486486487
Accuracy: 0.7567567567567568


##
---

## **Evaluating Cross-Validation Results**

After performing cross-validation, metrics such as accuracy, precision, recall, or F1-score are averaged across all folds.

- **cross_val_score(model, X, y, cv=cv)** : return the score values for each datasets.
  - *model* : Machine learning model
  - *X* : 2D `x` values of the dataset.
  - *y* : target (y) values of the dataset.
  - *cv* : Number of split data sets.

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

# Load dataset
data = load_iris()
X, y = data.data, data.target

model = RandomForestClassifier()

scores = cross_val_score(model, X, y, cv=5)
print("Accuracy per fold:", scores)
print("Mean Accuracy:", scores.mean())

Accuracy per fold: [0.96666667 0.96666667 0.93333333 0.96666667 1.        ]
Mean Accuracy: 0.9666666666666668


##
---

## **Advantages & Disadvantages of Cross-Validation**

### Advantages of Cross-Validation

- Efficient use of data for training and testing.

- Provides more reliable performance estimates.

- Reduces variance in evaluation compared to a single train-test split.



### Disadvantages of Cross-Validation

- Computationally Intensive:

  - Requires training the model multiple times.

- Complexity:

  - Difficult to implement in some scenarios like time-series data.

- Bias-Variance Tradeoff:

  - LOOCV has high variance, while K-Fold might introduce some bias.

##
---

## **Mathematical Equations**

For K-Fold Cross-Validation, the average score across all folds is calculated as:

$$
\text{Score}_{\text{avg}} = \frac{1}{K} \sum_{i=1}^{K} \text{Score}_i
$$

Where:

- K = Number of folds.

- $\text{Score}_i$​ = Performance metric (e.g., accuracy) for the $i^{th}$ fold.

##
---

## **Comparison of Cross-Validation Methods**

| Method              |   Strengths                          | Weaknesses                                                    |
|---------------------|--------------------------------------|---------------------------------------------------------------|
|K-Fold               | Reliable, balances bias and variance |Computational cost for large datasets                          |
|Stratified K-Fold    | Handles imbalanced datasets          |Similar to K-Fold limitations                                  |
|Leave-One-Out (LOOCV) | Best for small datasets             |Very computationally expensive                                 |
|Hold-Out Validation  |Simple and fast                       |Results depend on the split                                    |
|Time Series          |Respects time order                   |Limited to sequential data                                     |

##
---

Cross-validation is a cornerstone technique in machine learning that ensures robust and unbiased evaluation of model performance, making it essential for any ML workflow.

##
----